In [1]:
# Nếu chạy Colab L4, cell này chạy 1 lần là đủ.
!pip -q install pandas numpy scikit-learn tqdm torch transformers accelerate bitsandbytes sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.7 MB/s eta 0:00:00


In [2]:
!pip install -U huggingface_hub hf_transfer
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 92.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0


In [3]:
import os
import re
import json
import random
import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from sklearn.metrics import cohen_kappa_score, mean_absolute_error, mean_squared_error

import torch
import torch.nn as nn

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer


In [4]:
# Reproducibility
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

In [5]:
TRAIN_PATH = "/content/asap_train.csv"
VAL_PATH   = "/content/asap_val.csv"
TEST_PATH  = "/content/asap_test.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

required_cols = ["essay_id", "essay_set", "essay", "domain1_score", "domain1_score_norm"]

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    missing = set(required_cols) - set(df.columns)
    assert not missing, f"{name} missing columns: {missing}"

train_df.head()

,essay_id,essay_set,essay,domain1_score,domain1_score_norm
0,14876,6,In the excerpt The Mooring Mast by Marcia Amid...,3.0,0.75
1,9985,4,The author concluded this sentence because he ...,0.0,0.00
2,3231,2,"I can think of several books that, I would not...",4.0,0.60
3,21137,8,My best friend @PERSON2 turned thirteen on a b...,39.0,0.65
4,17919,7,A time that I was patient is when I broke my f...,23.0,0.77


In [6]:
ESSAY_SET = 2

p1_train = train_df[train_df["essay_set"] == ESSAY_SET].copy().reset_index(drop=True)
p1_val   = val_df[val_df["essay_set"] == ESSAY_SET].copy().reset_index(drop=True)
p1_test  = test_df[test_df["essay_set"] == ESSAY_SET].copy().reset_index(drop=True)

print("P1 train:", p1_train.shape)
print("P1 val:  ", p1_val.shape)
print("P1 test: ", p1_test.shape)

p1_train[["essay_id", "essay_set", "domain1_score", "domain1_score_norm"]].head()

P1 train: (1260, 5)
P1 val:   (180, 5)
P1 test:  (360, 5)


,essay_id,essay_set,domain1_score,domain1_score_norm
0,3231,2,4.0,0.6
1,4244,2,4.0,0.6
2,4046,2,4.0,0.6
3,4344,2,4.0,0.6
4,3531,2,4.0,0.6


In [7]:
ASAP_SCORE_RANGES = {
    1: (2, 12),
    2: (1, 6),
    3: (0, 3),
    4: (0, 3),
    5: (0, 4),
    6: (0, 4),
    7: (0, 30),
    8: (0, 60),
}

Y_MIN, Y_MAX = ASAP_SCORE_RANGES[ESSAY_SET]
Y_MIN, Y_MAX

(1, 6)

In [8]:
def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    spearman = pd.Series(y_true).corr(pd.Series(y_pred), method="spearman")

    return {
        "QWK": float(qwk),
        "MAE": float(mae),
        "RMSE": float(rmse),
        "Spearman": float(spearman) if not pd.isna(spearman) else np.nan,
    }


In [9]:
def sample_pairs(df, num_pairs=1000, seed=42, min_per_essay=3):
    rng = np.random.default_rng(seed)
    essay_ids = df["essay_id"].astype(int).tolist()

    pairs = set()
    counts = {eid: 0 for eid in essay_ids}

    # 1. Ensure each training essay appears at least min_per_essay times if possible
    attempts = 0
    max_attempts = num_pairs * 200

    while min(counts.values()) < min_per_essay and len(pairs) < num_pairs and attempts < max_attempts:
        underrepresented = [eid for eid, c in counts.items() if c < min_per_essay]

        a = int(rng.choice(underrepresented))
        b = int(rng.choice([eid for eid in essay_ids if eid != a]))

        x, y = sorted((a, b))

        if (x, y) not in pairs:
            pairs.add((x, y))
            counts[x] += 1
            counts[y] += 1

        attempts += 1

    # 2. Fill remaining pairs randomly
    while len(pairs) < num_pairs:
        a, b = rng.choice(essay_ids, size=2, replace=False)
        x, y = sorted((int(a), int(b)))
        pairs.add((x, y))

    return pd.DataFrame(list(pairs), columns=["essay_id_1", "essay_id_2"])


# Inductive setup:
# - Generate pairwise comparisons ONLY among training essays.
# - Train RankNet ONLY on training essays.
# - Apply the trained scoring model to validation/test essays that were unseen
#   during pairwise comparison generation and RankNet training.
p1_train_ind = p1_train.copy()
p1_val_ind = p1_val.copy()
p1_test_ind = p1_test.copy()

p1_train_ind["split"] = "train"
p1_val_ind["split"] = "val"
p1_test_ind["split"] = "test"

# Keep the same config as the paper-like transductive notebook.
M_PAIRWISE = 7500
MIN_PER_ESSAY = 4

train_pairs = sample_pairs(
    p1_train_ind,
    num_pairs=M_PAIRWISE,
    seed=SEED,
    min_per_essay=MIN_PER_ESSAY,
)

print("P1 train:", p1_train_ind.shape)
print("P1 val:  ", p1_val_ind.shape)
print("P1 test: ", p1_test_ind.shape)
print("Inductive train-only pairwise comparisons M:", train_pairs.shape)
train_pairs.head()


P1 train: (1260, 6)
P1 val:   (180, 6)
P1 test:  (360, 6)
Inductive train-only pairwise comparisons M: (7500, 2)


,essay_id_1,essay_id_2
0,3069,3705
1,4512,4618
2,3144,4522
3,3445,3696
4,4387,4623


In [10]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

# A100-friendly BF16 loading. If your runtime does not support BF16, switch
# torch_dtype to torch.float16. 4-bit is not needed for A100 and can be slower.
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"
llm.eval()

print("Loaded:", MODEL_ID)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-7B-Instruct


In [11]:
P1_RUBRIC_COMPACT = """
Evaluate the overall quality of a persuasive essay written to a newspaper about censorship in libraries.

This essay set is assessed using two trait domains:

Domain 1: Writing Applications
Prefer the essay that better:
- fully accomplishes the persuasive writing task,
- takes and maintains a clear position on censorship in libraries,
- supports the position with relevant, convincing, and well-developed reasons, examples, observations, or reading,
- stays focused on the topic and avoids tangents,
- presents a unifying main idea or theme,
- is logically organized with a clear beginning, middle, and end,
- uses transitions and topic sentences to connect ideas,
- has an effective introduction and conclusion,
- uses precise and appropriate vocabulary,
- demonstrates fluency and sentence variety,
- adjusts language and tone appropriately for a newspaper audience,
- shows a clear sense of audience and, when possible, an original perspective.

Domain 2: Language Conventions
Also consider which essay shows better control of:
- capitalization,
- punctuation,
- spelling,
- grammar and Standard English usage,
- paragraphing,
- sentence structure,
- avoidance of run-ons and fragments.

When making the final pairwise choice, prioritize Domain 1 writing quality, but use Domain 2 language conventions as an important secondary factor.
Choose "tie" only if the two essays are genuinely very close in overall quality.
""".strip()

In [12]:
P1_PROMPT_TEXT = """
Censorship in the Libraries

"All of us can think of a book that we hope none of our children or any other children have taken off the shelf.
But if I have the right to remove that book from the shelf -- that work I abhor -- then you also have exactly
the same right and so does everyone else. And then we have no books left on the shelf for any of us."
-- Katherine Paterson, Author

Write a persuasive essay to a newspaper reflecting your views on censorship in libraries.
Do you believe that certain materials, such as books, music, movies, magazines, etc.,
should be removed from the shelves if they are found offensive?

Support your position with convincing arguments from your own experience, observations, and/or reading.
""".strip()

In [13]:
SYSTEM_PROMPT_ASAP = """
As an English teacher, your primary responsibility is to evaluate the writing quality of essays written by middle school students on an English exam.
During the assessment process, you will be provided with a prompt, rubric guidelines, and two essays.
Determine which essay, Essay 1 or Essay 2, would receive a higher overall score, or respond with tie if they would receive the same score.
""".strip()


def build_pairwise_prompt(essay1, essay2):
    """Paper-like LCES prompt for ASAP pairwise comparison.

    The paper uses a teacher role, rubric, an anonymization note, brief
    reasoning, and a final JSON decision. The parser below is robust so that
    malformed JSON does not automatically collapse the pipeline.
    """
    return f"""
# Prompt
{P1_PROMPT_TEXT}

# Rubric Guidelines
{P1_RUBRIC_COMPACT}

# Note
The essays may contain anonymized entities such as PERSON, ORGANIZATION, LOCATION, DATE, TIME, MONEY, PERCENT, CAPS, or NUM.
Do not penalize the essay because of anonymization.

# Essay1
{essay1}

# Essay2
{essay2}

Provide your reasoning and final decision in JSON format:
{{"reasoning": "Your reasoning in one concise sentence.", "preference": "essay1" or "essay2" or "tie"}}
""".strip()


In [14]:
def extract_preference(text):
    """Extract preference from paper-like JSON output, with robust fallbacks."""
    raw = text
    text = text.strip()

    def normalize_pref(pref):
        pref = str(pref).lower().strip()
        pref = pref.replace(" ", "")
        pref = pref.replace("_", "")
        pref = pref.replace('"', "")
        pref = pref.replace("'", "")
        if pref in ["essay1", "1"]:
            return "essay1"
        if pref in ["essay2", "2"]:
            return "essay2"
        if pref in ["tie", "equal", "same", "samequality", "both"]:
            return "tie"
        return None

    # 1. Direct JSON parse.
    try:
        obj = json.loads(text)
        pref = normalize_pref(obj.get("preference", ""))
        if pref is not None:
            return {
                "reasoning": str(obj.get("reasoning", ""))[:500],
                "preference": pref,
                "raw_text": raw[:2000],
            }
    except Exception:
        pass

    # 2. Extract first JSON-looking block.
    match = re.search(r"\{.*?\}", text, flags=re.DOTALL)
    if match:
        try:
            obj = json.loads(match.group(0))
            pref = normalize_pref(obj.get("preference", ""))
            if pref is not None:
                return {
                    "reasoning": str(obj.get("reasoning", ""))[:500],
                    "preference": pref,
                    "raw_text": raw[:2000],
                }
        except Exception:
            pass

    # 3. Regex preference field from imperfect JSON/text.
    pref_match = re.search(
        r'["\']?preference["\']?\s*[:=]\s*["\']?(essay\s*1|essay\s*2|essay1|essay2|tie)',
        text,
        flags=re.IGNORECASE,
    )
    if pref_match:
        pref = normalize_pref(pref_match.group(1))
        if pref is not None:
            return {"reasoning": text[:500], "preference": pref, "raw_text": raw[:2000]}

    # 4. Final-decision style fallback.
    decision_match = re.search(
        r'(decision|final decision|answer)\s*[:=\-]?\s*(essay\s*1|essay\s*2|essay1|essay2|tie)',
        text,
        flags=re.IGNORECASE,
    )
    if decision_match:
        pref = normalize_pref(decision_match.group(2))
        if pref is not None:
            return {"reasoning": text[:500], "preference": pref, "raw_text": raw[:2000]}

    # 5. Last-resort: inspect the tail only. This avoids picking up Essay1/Essay2
    # from the prompt if the model echoes part of the input.
    tail = text[-500:].lower()
    if re.search(r"\bessay\s*1\b|\bessay1\b", tail):
        pref = "essay1"
    elif re.search(r"\bessay\s*2\b|\bessay2\b", tail):
        pref = "essay2"
    elif re.search(r"\btie\b|same score|equal quality|equally", tail):
        pref = "tie"
    else:
        # Parser failure should be rare. Returning tie is still consistent with
        # LCES, but inspect raw_text if many rows fall here.
        pref = "tie"

    return {"reasoning": text[:500], "preference": pref, "raw_text": raw[:2000]}


tokenizer.padding_side = "left"

@torch.inference_mode()
def call_local_llm(prompts, max_new_tokens=64, temperature=0.1):
    """Batched local LLM inference for pairwise judgments."""
    input_texts = []

    for prompt in prompts:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT_ASAP},
            {"role": "user", "content": prompt},
        ]

        if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template is not None:
            input_text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        else:
            input_text = SYSTEM_PROMPT_ASAP + "\n\n" + prompt

        input_texts.append(input_text)

    inputs = tokenizer(
        input_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048,
    ).to(llm.device)

    do_sample = temperature is not None and temperature > 0

    output_ids = llm.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[:, prompt_len:]

    texts = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    results = [extract_preference(t) for t in texts]

    del inputs, output_ids, generated_ids

    return results


In [15]:
# Quick smoke test on training essays only
sample_essay_1 = p1_train_ind.iloc[0]["essay"]
sample_essay_2 = p1_train_ind.iloc[1]["essay"]

test_prompt = build_pairwise_prompt(sample_essay_1, sample_essay_2)
call_local_llm([test_prompt], max_new_tokens=64, temperature=0.1)


[{'reasoning': 'Essay 2 provides a more balanced argument, addresses various forms of media, and maintains a clearer focus on the topic without digressing into personal anecdotes. It also demonstrates a better understanding of the issue and a more appropriate tone for a newspaper audience.',
  'preference': 'essay2',
  'raw_text': '{"reasoning": "Essay 2 provides a more balanced argument, addresses various forms of media, and maintains a clearer focus on the topic without digressing into personal anecdotes. It also demonstrates a better understanding of the issue and a more appropriate tone for a newspaper audience.", "preference": "essay2"}'}]

In [16]:
def preference_to_label(preference):
    pref = str(preference).lower().strip()
    pref = pref.replace(" ", "")
    pref = pref.replace("_", "")
    pref = pref.replace('"', "")
    pref = pref.replace("'", "")

    if pref in ["essay1", "1"]:
        return 1.0
    if pref in ["essay2", "2"]:
        return 0.0
    if pref in ["tie", "equal", "same", "both"]:
        return 0.5

    return 0.5

def debias_label(label_forward, label_reverse):
    """
    Forward: essay_id_1 as Essay 1, essay_id_2 as Essay 2.
    Reverse: essay_id_2 as Essay 1, essay_id_1 as Essay 2.

    If consistent:
      label_forward == 1 - label_reverse

    Else:
      tie.
    """
    if np.isclose(label_forward, 1.0 - label_reverse):
        return label_forward
    return 0.5

In [17]:
def generate_pairwise_judgments(
    df,
    pairs_df,
    debias=True,
    batch_size=16,
    max_new_tokens=64,
    temperature=0.1,
):
    """Generate pairwise LLM judgments with LCES position-bias correction."""
    essay_map = df.set_index("essay_id")["essay"].to_dict()
    rows = []

    pair_rows = list(pairs_df.itertuples(index=False))

    for start in tqdm(range(0, len(pair_rows), batch_size)):
        batch = pair_rows[start:start + batch_size]

        forward_prompts = []
        reverse_prompts = []
        meta = []

        for row in batch:
            id1 = int(row.essay_id_1)
            id2 = int(row.essay_id_2)

            essay1 = essay_map[id1]
            essay2 = essay_map[id2]

            forward_prompts.append(build_pairwise_prompt(essay1, essay2))

            if debias:
                reverse_prompts.append(build_pairwise_prompt(essay2, essay1))

            meta.append((id1, id2))

        forward_results = call_local_llm(
            forward_prompts,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
        )

        if debias:
            reverse_results = call_local_llm(
                reverse_prompts,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
            )
        else:
            reverse_results = [None] * len(forward_results)

        for (id1, id2), result_forward, result_reverse in zip(meta, forward_results, reverse_results):
            label_forward = preference_to_label(result_forward.get("preference", "tie"))

            if debias:
                label_reverse = preference_to_label(result_reverse.get("preference", "tie"))
                final_label = debias_label(label_forward, label_reverse)

                rows.append({
                    "essay_id_1": id1,
                    "essay_id_2": id2,
                    "label": final_label,
                    "pref_forward": result_forward.get("preference", "tie"),
                    "pref_reverse": result_reverse.get("preference", "tie"),
                    "reason_forward": result_forward.get("reasoning", ""),
                    "reason_reverse": result_reverse.get("reasoning", ""),
                    "raw_forward": result_forward.get("raw_text", ""),
                    "raw_reverse": result_reverse.get("raw_text", ""),
                })
            else:
                rows.append({
                    "essay_id_1": id1,
                    "essay_id_2": id2,
                    "label": label_forward,
                    "pref_forward": result_forward.get("preference", "tie"),
                    "pref_reverse": None,
                    "reason_forward": result_forward.get("reasoning", ""),
                    "reason_reverse": None,
                    "raw_forward": result_forward.get("raw_text", ""),
                    "raw_reverse": None,
                })

        del batch, forward_prompts, reverse_prompts, forward_results, reverse_results, meta

        batch_idx = start // batch_size
        if torch.cuda.is_available() and batch_idx % 25 == 0:
            torch.cuda.empty_cache()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return pd.DataFrame(rows)


def inspect_judgments(judgments):
    print("pref_forward")
    print(judgments["pref_forward"].value_counts(dropna=False))

    if "pref_reverse" in judgments.columns:
        print("\npref_reverse")
        print(judgments["pref_reverse"].value_counts(dropna=False))

    print("\nlabel")
    print(judgments["label"].value_counts(dropna=False))

    tie_rate = (judgments["label"] == 0.5).mean()
    print(f"\nTie rate: {tie_rate:.2%}")

    if tie_rate > 0.60:
        raise ValueError("Tie rate is high. Inspect raw outputs before training RankNet.")


### Pairwise judgment generation
The following cell generates or loads cached LCES pairwise comparisons for the **inductive** Prompt 1 setup. Comparisons are generated only among training essays; validation and test essays are not included in the comparison graph.

In [18]:
JUDGMENT_CACHE = f"train_judgments_p2_qwen25_7b_paperprompt_m{M_PAIRWISE}_min{MIN_PER_ESSAY}_debias_inductive.csv"


if os.path.exists(JUDGMENT_CACHE):
    train_judgments = pd.read_csv(JUDGMENT_CACHE)
    print("Loaded cached judgments:", JUDGMENT_CACHE)
else:
    train_judgments = generate_pairwise_judgments(
        df=p1_train_ind,
        pairs_df=train_pairs,
        debias=True,
        batch_size=16,
        max_new_tokens=64,
        temperature=0.1,
    )

    train_judgments.to_csv(JUDGMENT_CACHE, index=False)
    print("Saved judgments:", JUDGMENT_CACHE)

inspect_judgments(train_judgments)

# Free LLM VRAM before loading the embedding model and training RankNet.
if "llm" in globals():
    del llm
if "tokenizer" in globals():
    del tokenizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()


  0%|          | 0/469 [00:00<?, ?it/s]

Saved judgments: train_judgments_p2_qwen25_7b_paperprompt_m7500_min4_debias_inductive.csv
pref_forward
pref_forward
essay2    5055
essay1    2251
tie        194
Name: count, dtype: int64

pref_reverse
pref_reverse
essay2    5017
essay1    2287
tie        196
Name: count, dtype: int64

label
label
0.5    3226
0.0    2156
1.0    2118
Name: count, dtype: int64

Tie rate: 43.01%


In [19]:
used_ids = set(train_judgments["essay_id_1"]) | set(train_judgments["essay_id_2"])

print("Total P1 train essays:", len(p1_train_ind))
print("Train essays appearing in comparisons:", len(used_ids))
print("Train coverage:", len(used_ids) / len(p1_train_ind))

pair_count = pd.concat([
    train_judgments["essay_id_1"],
    train_judgments["essay_id_2"],
]).value_counts()

print("\nPair count per training essay:")
print(pair_count.describe())

# Sanity check: validation/test essays should not appear in inductive comparisons.
val_test_ids = set(p1_val_ind["essay_id"].astype(int)) | set(p1_test_ind["essay_id"].astype(int))
leaked_ids = used_ids & val_test_ids
print("\nVal/test essays in comparison graph:", len(leaked_ids))
assert len(leaked_ids) == 0, "Inductive violation: val/test essays appeared in train comparisons."


Total P1 train essays: 1260
Train essays appearing in comparisons: 1260
Train coverage: 1.0

Pair count per training essay:
count    1260.000000
mean       11.904762
std         2.718537
min         4.000000
25%        10.000000
50%        12.000000
75%        14.000000
max        21.000000
Name: count, dtype: float64

Val/test essays in comparison graph: 0


In [20]:
# Paper main experiments use OpenAI text-embedding-3-large. To keep this
# notebook runnable without an API key, we use RoBERTa-base [CLS], which the
# paper also reports as a robust alternative in Appendix B.
EMBEDDING_MODEL = "roberta-base"

embedding_tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)
embedding_model = AutoModel.from_pretrained(EMBEDDING_MODEL).to(DEVICE)
embedding_model.eval()

@torch.inference_mode()
def encode_essays_cls(df, text_col="essay", batch_size=16, max_length=512):
    all_vecs = []
    texts = df[text_col].tolist()

    for start in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[start:start + batch_size]

        inputs = embedding_tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
        ).to(DEVICE)

        outputs = embedding_model(**inputs)
        cls_vec = outputs.last_hidden_state[:, 0, :]
        all_vecs.append(cls_vec.detach().cpu().numpy())

        del inputs, outputs, cls_vec

    return np.vstack(all_vecs).astype(np.float32)


train_embeddings = encode_essays_cls(p1_train_ind, batch_size=16, max_length=512)
val_embeddings = encode_essays_cls(p1_val_ind, batch_size=16, max_length=512)
test_embeddings = encode_essays_cls(p1_test_ind, batch_size=16, max_length=512)

print("Embedding model:", EMBEDDING_MODEL)
print("train_embeddings:", train_embeddings.shape)
print("val_embeddings:  ", val_embeddings.shape)
print("test_embeddings: ", test_embeddings.shape)


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  0%|          | 0/79 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/23 [00:00<?, ?it/s]

Embedding model: roberta-base
train_embeddings: (1260, 768)
val_embeddings:   (180, 768)
test_embeddings:  (360, 768)


In [21]:
class RankNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, dropout=0.3):
        super().__init__()

        self.scorer = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.scorer(x).squeeze(-1)

In [22]:
def train_ranknet(
    df,
    embeddings,
    judgments,
    epochs=100,
    batch_size=4096,
    lr=1e-3,
    weight_decay=0.01,
    hidden_dim=256,
    dropout=0.3,
    device=DEVICE
):
    essay_ids = df["essay_id"].astype(int).tolist()
    id_to_idx = {eid: idx for idx, eid in enumerate(essay_ids)}

    pair_df = judgments[
        judgments["essay_id_1"].isin(id_to_idx) &
        judgments["essay_id_2"].isin(id_to_idx)
    ].copy()

    assert len(pair_df) > 0, "No valid pairwise judgments for this dataframe."

    idx_i = pair_df["essay_id_1"].map(id_to_idx).values
    idx_j = pair_df["essay_id_2"].map(id_to_idx).values
    labels = pair_df["label"].values.astype(np.float32)

    x = torch.tensor(embeddings, dtype=torch.float32, device=device)

    model = RankNet(
        input_dim=embeddings.shape[1],
        hidden_dim=hidden_dim,
        dropout=dropout
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    loss_fn = nn.BCELoss()

    n = len(pair_df)

    for epoch in range(1, epochs + 1):
        order = np.random.permutation(n)
        epoch_losses = []

        model.train()

        for start in range(0, n, batch_size):
            batch = order[start:start + batch_size]

            bi = torch.tensor(idx_i[batch], dtype=torch.long, device=device)
            bj = torch.tensor(idx_j[batch], dtype=torch.long, device=device)
            y = torch.tensor(labels[batch], dtype=torch.float32, device=device)

            s_i = model(x[bi])
            s_j = model(x[bj])

            pred = torch.sigmoid(s_i - s_j)
            loss = loss_fn(pred, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_losses.append(loss.item())

        if epoch == 1 or epoch % 10 == 0:
            print(f"Epoch {epoch:03d} | loss = {np.mean(epoch_losses):.4f}")

    return model

In [23]:
# Paper hyperparameters: epochs=100, batch_size=4096, hidden_dim=256,
# dropout=0.3, weight_decay=0.01, lr=0.001.
# Inductive: train RankNet only on training essays and train-only judgments.
ranknet = train_ranknet(
    df=p1_train_ind,
    embeddings=train_embeddings,
    judgments=train_judgments,
    epochs=100,
    batch_size=4096,
    lr=1e-3,
    weight_decay=0.01,
    hidden_dim=256,
    dropout=0.3,
)


Epoch 001 | loss = 0.6917
Epoch 010 | loss = 0.6505
Epoch 020 | loss = 0.6044
Epoch 030 | loss = 0.5824
Epoch 040 | loss = 0.5694
Epoch 050 | loss = 0.5615
Epoch 060 | loss = 0.5570
Epoch 070 | loss = 0.5549
Epoch 080 | loss = 0.5510
Epoch 090 | loss = 0.5501
Epoch 100 | loss = 0.5487


In [24]:
@torch.no_grad()
def predict_latent_scores(model, embeddings, device=DEVICE):
    model.eval()
    x = torch.tensor(embeddings, dtype=torch.float32, device=device)
    scores = model(x).detach().cpu().numpy()
    return scores


train_latent = predict_latent_scores(ranknet, train_embeddings)
val_latent = predict_latent_scores(ranknet, val_embeddings)
test_latent = predict_latent_scores(ranknet, test_embeddings)

print("train latent min:", train_latent.min())
print("train latent max:", train_latent.max())
print("train latent std:", train_latent.std())
print("val latent std:  ", val_latent.std())
print("test latent std: ", test_latent.std())
print("train latent head:", train_latent[:10])


train latent min: -2.8255086
train latent max: 0.95811915
train latent std: 0.7029877
val latent std:   0.69710803
test latent std:  0.6724385
train latent head: [-4.8046401e-01 -8.0138334e-04  3.5435024e-01  8.4303088e-02
 -7.5700678e-02 -2.8628820e-01 -1.7372426e+00 -2.1342974e+00
 -9.5579606e-01 -1.0513961e+00]


In [25]:
def fit_latent_mapper(latent_scores_ref, y_min, y_max):
    """Paper's linear latent-score conversion to the rubric range.

    Inductive version: fit the conversion using training latent scores only,
    then apply the same mapper to validation/test latent scores.
    """
    s_min = float(np.min(latent_scores_ref))
    s_max = float(np.max(latent_scores_ref))

    def mapper(latent_scores):
        latent_scores = np.asarray(latent_scores, dtype=float)

        if np.isclose(s_min, s_max):
            mapped = np.full_like(latent_scores, fill_value=(y_min + y_max) / 2)
        else:
            mapped = (latent_scores - s_min) / (s_max - s_min)
            mapped = mapped * (y_max - y_min) + y_min

        mapped = np.rint(mapped)
        mapped = np.clip(mapped, y_min, y_max)

        return mapped.astype(int)

    return mapper


score_mapper = fit_latent_mapper(train_latent, Y_MIN, Y_MAX)

p1_train_pred = score_mapper(train_latent)
p1_val_pred = score_mapper(val_latent)
p1_test_pred = score_mapper(test_latent)

p1_train_scored = p1_train_ind[[
    "essay_id", "split", "essay_set", "essay", "domain1_score", "domain1_score_norm"
]].copy()
p1_val_scored = p1_val_ind[[
    "essay_id", "split", "essay_set", "essay", "domain1_score", "domain1_score_norm"
]].copy()
p1_test_scored = p1_test_ind[[
    "essay_id", "split", "essay_set", "essay", "domain1_score", "domain1_score_norm"
]].copy()

p1_train_scored["latent_score"] = train_latent
p1_val_scored["latent_score"] = val_latent
p1_test_scored["latent_score"] = test_latent

p1_train_scored["pred_score"] = p1_train_pred
p1_val_scored["pred_score"] = p1_val_pred
p1_test_scored["pred_score"] = p1_test_pred

p1_all_scored = pd.concat(
    [p1_train_scored, p1_val_scored, p1_test_scored],
    axis=0,
    ignore_index=True,
)

p1_all_scored["abs_error"] = np.abs(
    p1_all_scored["domain1_score"] - p1_all_scored["pred_score"]
)

print("Train pred distribution:")
print(pd.Series(p1_train_pred).value_counts().sort_index())
print("\nVal pred distribution:")
print(pd.Series(p1_val_pred).value_counts().sort_index())
print("\nTest pred distribution:")
print(pd.Series(p1_test_pred).value_counts().sort_index())

p1_all_scored.head()


Train pred distribution:
1      8
2    105
3    331
4    492
5    294
6     30
Name: count, dtype: int64

Val pred distribution:
1     1
2    16
3    47
4    67
5    45
6     4
Name: count, dtype: int64

Test pred distribution:
1      2
2     21
3     89
4    149
5     87
6     12
Name: count, dtype: int64


,essay_id,split,essay_set,essay,domain1_score,domain1_score_norm,latent_score,pred_score,abs_error
0,3231,train,2,"I can think of several books that, I would not...",4.0,0.6,-0.480464,4,0.0
1,4244,train,2,I believe that everyone's opinion deserves to ...,4.0,0.6,-0.000801,5,1.0
2,4046,train,2,Do you want your children to have bad images i...,4.0,0.6,0.354350,5,1.0
3,4344,train,2,"In many @CAPS1 libraries across the nation, a ...",4.0,0.6,0.084303,5,1.0
4,3531,train,2,"One @MONTH1 walk into a library, look over at ...",4.0,0.6,-0.075701,5,1.0


In [26]:
metrics_rows = []

for split in ["train", "val", "test", "all"]:
    if split == "all":
        df_eval = p1_all_scored
    else:
        df_eval = p1_all_scored[p1_all_scored["split"] == split]

    metrics_rows.append({
        "split": split,
        **compute_metrics(
            y_true=df_eval["domain1_score"],
            y_pred=df_eval["pred_score"],
        ),
    })

pd.DataFrame(metrics_rows)


,split,QWK,MAE,RMSE,Spearman
0,train,0.580188,0.612698,0.851702,0.647010
1,val,0.589346,0.616667,0.846562,0.671892
2,test,0.539477,0.652778,0.902158,0.632552
3,all,0.572764,0.621111,0.861523,0.646119


In [27]:
p1_test_results = p1_all_scored[p1_all_scored["split"] == "test"].copy()

p1_test_results[[
    "essay_id",
    "domain1_score",
    "pred_score",
    "abs_error",
    "latent_score",
]].head(20)


,essay_id,domain1_score,pred_score,abs_error,latent_score
1440,4119,3.0,3,0.0,-1.289344
1441,4399,3.0,3,0.0,-1.019421
1442,3998,4.0,4,0.0,-0.227580
1443,3532,3.0,4,1.0,-0.710733
1444,4621,4.0,5,1.0,-0.168021
1445,3880,3.0,3,0.0,-1.175261
1446,3456,3.0,3,0.0,-1.267337
1447,3471,3.0,4,1.0,-0.730363
1448,3244,2.0,4,2.0,-0.601517
1449,4282,3.0,3,0.0,-1.247199


In [28]:
# Worst predictions on the held-out test split
p1_test_results.sort_values("abs_error", ascending=False)[[
    "essay_id",
    "domain1_score",
    "pred_score",
    "abs_error",
    "latent_score",
    "essay",
]].head(10)


,essay_id,domain1_score,pred_score,abs_error,latent_score,essay
1454,3180,3.0,5,2.0,-0.083564,"Sexual conduct, language, and criminal behavio..."
1719,4224,3.0,5,2.0,-0.050283,People go to the library to find certain types...
1488,3330,3.0,5,2.0,-0.125022,"Eventually, everyone sees or hears offensive t..."
1498,3915,3.0,5,2.0,-0.076216,"Certain material such as books, movies, and ma..."
1727,4615,3.0,5,2.0,-0.030634,Communities have many different types of peopl...
1725,2996,1.0,3,2.0,-1.526755,If the people that are publishing and writing ...
1533,3169,3.0,5,2.0,0.335649,'And then we have no books left on the shelf f...
1525,3534,3.0,5,2.0,-0.140494,"No, I don't think that books or any other mate..."
1726,3822,3.0,5,2.0,0.022971,Libraries are often known for protecting our c...
1792,4585,3.0,5,2.0,0.025962,"Yes, some materials,such as books, music, movi..."


In [30]:
from google.colab import files, runtime
import os
import shutil
import time

# Tên file CSV đã save
FILENAME = f"train_judgments_p2_qwen25_7b_paperprompt_m{M_PAIRWISE}_min{MIN_PER_ESSAY}_debias_inductive.csv"

# Đường dẫn file cần tải
SAVE_PATH = f"/content/{FILENAME}"

# Nếu là thư mục thì zip rồi tải, nếu là file thì tải trực tiếp
if os.path.isdir(SAVE_PATH):
    ZIP_NAME = FILENAME.rstrip("/").replace(".csv", "") + ".zip"
    shutil.make_archive(ZIP_NAME.replace(".zip", ""), "zip", SAVE_PATH)
    files.download(ZIP_NAME)

elif os.path.isfile(SAVE_PATH):
    files.download(SAVE_PATH)

else:
    raise FileNotFoundError(f"Không tìm thấy file/thư mục: {SAVE_PATH}")

# Chờ tải bắt đầu rồi ngắt runtime
time.sleep(10)
runtime.unassign()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

RuntimeManagementError: Unable to request VM unassignment.